### Coarse database

In [ ]:
### database
!python /home/dmin/CoarseRL2Mol/spectra-scraper/src/curate_nmr_dataset.py

In [ ]:
### generate NMR spectra
!python /home/dmin/CoarseRL2Mol/spectra-scraper/src/generate_curated_nmr_spectra.py

In [ ]:
### draw SMILES 
import os
import sys
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display, SVG
import matplotlib.pyplot as plt
smiles = "C1=COC(=C1)C(=O)O"
name = " "
mol = Chem.MolFromSmiles(smiles)
img = Draw.MolToImage(mol, size=(400, 300))
plt.figure(figsize=(5, 4))
plt.imshow(img)
plt.title(name)
plt.axis('off')
plt.show()

In [1]:
### coarse-graining
!/home/dmin/miniconda/envs/NMRer/bin/python model/shared/coarse_graph.py \
  --smiles "C1=COC(=C1)C(=O)O" \
  --name molecule_big \
  --output_dir ./demo \
  --node_radius 0.28 \
  --font_size 15 \
  --label_rotation 0

输出目录: ./demo

分子: molecule_big
SMILES: C1=COC(=C1)C(=O)O
  -> 图数据字典已保存到: ./demo/molecule_big.pt
✅ 转换成功!
   粗粒度节点数: 6
   粗粒度边数: 6
   总原子组成 (C,H,O,N,S,X): [5.0, 4.0, 3.0, 0.0, 0.0, 0.0]

📊 结构单元(SU)分配:
   节点  0: SU 13 (Carbocyclic_Aro_CH       )
   节点  1: SU 13 (Carbocyclic_Aro_CH       )
   节点  2: SU 29 (Ether_O                  )
   节点  3: SU  9 (Keto_Substituted_Aro_C   )
   节点  4: SU 13 (Carbocyclic_Aro_CH       )
   节点  5: SU  1 (Carboxylic_Acid          )

🖼️  图片: ./demo/molecule_big.png
📦 数据: ./demo/molecule_big.pt


In [ ]:
### database build
!/home/dmin/miniconda/envs/NMRer/bin/python model/data/build_dataset.py \
  --curated_dir data/curated_shift_dataset \
  --output_dir data/coarse_graph_data \
  --num_workers 4 \
  --unit_intensity \
  --min_su_nodes 6 \
  --no_png

In [ ]:
### check .pt
import torch
from torch_geometric.data import Data

data = Data.from_dict(torch.load('data/coarse_graph_data/1.pt', weights_only=False))
print('x:', data.x.shape)
print('edge_index:', data.edge_index.shape)
print('su_type_indices:', data.su_type_indices.tolist())
print('real_ppm:', data.real_ppm.tolist())
print('real_intensity:', data.real_intensity.tolist())
print('y_spectrum:', data.y_spectrum.shape)
print('carbon nodes with shift:', int((data.is_carbon & (data.real_intensity > 0)).sum()))

### S2N model

In [ ]:
### data preparation
!python -m model.data.dataset_utils split_batch_merge \
  --data_dir /home/dmin/CoarseRL2Mol/data/coarse_graph_data \
  --output_dir /mnt/e/code/coarsedata_v1 \
  --target_node_counts 50 60 70 80 90 100 120 140 160 180 200 220 240 260 280 300 350 400 450 500 550 \
  --num_per_target 40 \
  --val_num_per_target 15 \
  --test_num_per_target 15 \
  --train_ratio 0.84 \
  --val_ratio 0.08 \
  --test_ratio 0.08 \
  --seed 42 \
  --merge_styles random high_aromatic high_aliphatic high_oxygen high_carbonyl high_unsaturated high_hetero \
  --style_strength 2.0

In [ ]:
### training model
!python -m model.forward.s2n.s2n_train \
  --train_dirs /mnt/e/code/data3-R/s2n_split_merged_v2/train_single /mnt/e/code/data3-R/s2n_split_merged_v2/train_merged \
  --val_dirs /mnt/e/code/data3-R/s2n_split_merged_v2/val_single /mnt/e/code/data3-R/s2n_split_merged_v2/val_merged \
  --test_dirs /mnt/e/code/data3-R/s2n_split_merged_v2/test_single /mnt/e/code/data3-R/s2n_split_merged_v2/test_merged \
  --output_dir checkpoints_s2n_v1 \
  --target_mode per_100c \
  --use_spectrum_stats \
  --merged_repeat 2 \
  --epochs 1 \
  --batch_size 24 \
  --lr 0.00005 \
  --w_absolute 0.35 \
  --w_element 0.25 \
  --w_total 0.20 \
  --w_group 0.30 \
  --w_high_count 0.45 \
  --w_balance 0.10 \
  --balance_min_carbon 80

In [ ]:
### evaluating model
!python -m model.forward.s2n.s2n_eval \
  --model_path checkpoints_s2n_v1/best_s2n_model.pt \
  --input_dirs /mnt/e/code/data3-R/s2n_split_merged/test_single /mnt/e/code/data3-R/s2n_split_merged/test_merged \
  --output_dir s2n_test_eval \
  --scale_bins 0 80 160 260 380 1000000000 \
  --scale_by both \
  --examples_per_scale 2

In [ ]:
### predicting with model
!python -m model.forward.s2n.s2n_predict \
  --model_path checkpoints_s2n_v1/best_s2n_model.pt \
  --spectrum_csv data/YM.csv \
  --elements "C=500,H=521,O=197,N=6,S=0,X=0" \
  --output_name test \
  --apply_mask

In [ ]:
# 1. 同一谱图，不同元素：测试元素分支是否生效
!python -m model.forward.s2n.s2n_predict --model_path checkpoints_s2n_v1/best_s2n_model.pt --spectrum_csv data/HSQ.csv --elements "C=230,H=200,O=30,N=2,S=2,X=0" --output_name HSQ_C230

!python -m model.forward.s2n.s2n_predict --model_path checkpoints_s2n_v1/best_s2n_model.pt --spectrum_csv data/HSQ.csv --elements "C=500,H=521,O=197,N=6,S=0,X=0" --output_name HSQ_C500

### G2S model 

In [ ]:
### training G2S model
!python -m model.forward.g2s.g2s_train \
  --train_dirs /mnt/e/code/data3-R/s2n_split_merged/train_single \
  --val_dirs /mnt/e/code/data3-R/s2n_split_merged/val_single \
  --test_dirs /mnt/e/code/data3-R/s2n_split_merged/test_single \
  --output_dir checkpoints_g2s_shift_k3 \
  --k_hop 3 \
  --hid 384 \
  --latent_dim 16 \
  --epochs 1 \
  --batch_size 32 \
  --lr 0.0001

In [ ]:
### 1/2/3/4-hop
!python -m model.forward.g2s.g2s_ablation \
  --train_dirs coarsedata_v1/train_single \
  --val_dirs coarsedata_v1/val_single \
  --test_dirs coarsedata_v1/test_single \
  --output_dir checkpoints_g2s_ablation \
  --hops 1 2 3 4 \
  --epochs 150 \
  --batch_size 32

In [ ]:
### evaluating G2S model
!python -m model.forward.g2s.g2s_eval \
  --model_path checkpoints_g2s_shift_k3/best_g2s_model.pt \
  --input_dirs /mnt/e/code/data3-R/s2n_split_merged/test_single \
  --output_dir g2s_test_eval

In [ ]:
!python -m model.forward.g2s.g2s_predict \
  --model_path checkpoints_g2s_ablation/k3/best_g2s_model.pt \
  --input_path /mnt/e/code/coarsedata_v1/test_single/15721.pt \
  --output_dir checkpoints_g2s_ablation/g2s_single_15721 \
  --hwhm 1.0